## 03 - Data Cleaning & Feature Engineering

**Inputs:** `../reports/data_profile.json` (Notebook 01) and
`../reports/eda_summary.json` (Notebook 02) every decision below traces
back to a specific finding in one of those two files, cited by section
number, rather than being re-justified from scratch.

**A rule this notebook follows throughout, stated up front:** any
transformation whose parameters are *learned from the data's distribution*
(imputation values, one-hot vocabularies, scaling mean/std, PCA components,
target encoding) is **not fit here**. Fitting those on the full dataset
before the train/validation/test split leaks test-set statistics into
training. Those steps are deferred to a `scikit-learn` `Pipeline` /
`ColumnTransformer` built in `08_train_validation_test_split` /
`09_feature_selection`, fit on the training split only.

What *is* done here is everything safe to do before the split: dropping a
column, dropping duplicate rows, applying a fixed domain-knowledge mapping,
or computing a deterministic formula (like a sine/cosine transform of a
calendar field) none of these depend on which rows end up in train vs.
test, so doing them now doesn't leak anything.

## 0. Environment & Reproducibility

In [14]:
import sys
import json
import hashlib
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning, module="scipy")

import numpy as np
import pandas as pd

# Make the project's src/ package importable from inside notebooks/
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import engineer_features  # shared with models/predict.py

print(f"Python : {sys.version.split()[0]}")
print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


Python : 3.13.13
pandas : 2.3.3
numpy  : 2.5.2


## 1. Load Data + Inherited Artifacts

Nothing about column roles, the leakage finding, or the sentinel is
re-derived all of it is imported from the two upstream notebooks.

In [15]:
PROFILE_PATH = Path("../reports/data_profile.json")
EDA_PATH = Path("../reports/eda_summary.json")
RAW_PATH = Path("../data/raw/bank-additional-full.csv")

for p in (PROFILE_PATH, EDA_PATH, RAW_PATH):
    if not p.exists():
        raise FileNotFoundError(f"{p} not found. Run Notebooks 01 and 02 first.")

with open(PROFILE_PATH) as f:
    profile = json.load(f)
with open(EDA_PATH) as f:
    eda = json.load(f)

file_hash = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
if file_hash != profile["source_sha256"]:
    raise ValueError("Raw data file has changed since Notebook 01 profiled it.")

df = pd.read_csv(RAW_PATH, sep=";")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

target_col = profile["target_col"]
categorical_cols = list(profile["categorical_cols"])
leakage_cols = profile["leakage_cols"]
sentinels = profile["sentinels"]

cleaning_log = []   # every destructive/structural decision recorded here
fe_log = []          # every new column recorded here, with rationale

def log_clean(step, detail):
    cleaning_log.append({"step": step, "detail": detail})
    print(f"[CLEAN] {step}: {detail}")

def log_fe(column, dtype, rationale):
    fe_log.append({"column": column, "dtype": dtype, "rationale": rationale})
    print(f"[FEATURE] {column} ({dtype}): {rationale}")

Loaded: 41,188 rows x 21 columns


## 2. Data Cleaning

### 2.1 Drop the Leakage Column

`duration` was proven to leak the target (Notebook 01, Section 6;
Notebook 02, Section 3.4: correlation 0.41, `duration == 0` always maps to
`"no"`). It has been carried through Notebooks 01-02 for reference only
this is the point it actually gets dropped.

In [16]:
before_cols = df.shape[1]
df = df.drop(columns=leakage_cols)
log_clean("drop_leakage_columns", f"Dropped {leakage_cols}. Columns {before_cols} -> {df.shape[1]}.")

[CLEAN] drop_leakage_columns: Dropped ['duration']. Columns 21 -> 20.


### 2.2 Resolve the Duplicate-Rows Question

Notebook 01 (Section 8) left this open deliberately and ran the evidence
check in Notebook 01 v2: rows that are identical on every column except
`duration` have a target positive rate of **2.5%**, versus **12%** for the
rest of the data a ~5x gap that argues against "coincidental distinct
customers" and for "logging duplicates." That evidence is acted on now.

In [17]:
dupe_mask = df.duplicated(keep="first")
n_dupes = int(dupe_mask.sum())
before_rows = df.shape[0]

rate_in_dupes = df.loc[df.duplicated(keep=False), target_col].eq("yes").mean()
rate_rest = df.loc[~df.duplicated(keep=False), target_col].eq("yes").mean()
print(f"Target positive rate in duplicate rows : {rate_in_dupes:.1%}")
print(f"Target positive rate in the rest        : {rate_rest:.1%}")

df = df.loc[~dupe_mask].reset_index(drop=True)
log_clean("drop_duplicate_rows",
           f"Dropped {n_dupes} duplicate rows (excl. duration, keep=first) based on the "
           f"{rate_in_dupes:.1%} vs {rate_rest:.1%} target-rate evidence from Notebook 01. "
           f"Rows {before_rows:,} -> {df.shape[0]:,}.")

Target positive rate in duplicate rows : 2.5%
Target positive rate in the rest        : 12.0%
[CLEAN] drop_duplicate_rows: Dropped 1784 duplicate rows (excl. duration, keep=first) based on the 2.5% vs 12.0% target-rate evidence from Notebook 01. Rows 41,188 -> 39,404.


### 2.3 Neutralise the `pdays` Sentinel Then Drop Raw `pdays`

Three findings from Notebooks 01-02 combine into one decision here:

1. `pdays == 999` means "never contacted," corrupting every raw statistic
   on the column (Notebook 02, Section 1.1).
2. `previous` and `poutcome` agree with each other on **100%** of rows,
   while `pdays`'s sentinel disagrees with both on 4,110 rows (Notebook 01
   v2, Section 7) `pdays` is the *less* reliable of the three signals of
   prior contact, not a tie-breaker between them.
3. Real `pdays` values only exist for 3.7% of rows (Notebook 02, Section
   1.1) too sparse to carry much information on its own.

So: build the prior-contact flag from `previous` (the reliable signal),
keep the sparse real day-count as a supplementary numeric column for
feature selection to accept or reject downstream, and drop the raw,
sentinel-corrupted `pdays` column entirely.

### 2.4 `"unknown"` Categorical Values - Kept, Not Imputed

Notebook 02 (Section 6.2) found `default`, `education`, `housing`,
`loan`, `job`, and `marital` each have an `"unknown"` level, and that
`default`'s chi-square association with the target was statistically
significant. Imputing `"unknown"` to the mode would throw away a
potentially real signal (non-disclosure correlating with outcome) and,
worse, would require a fitted rule (the mode) that must not be computed on
data the model hasn't been trained on. **Decision: `"unknown"` stays as
its own explicit category**; the one-hot vocabulary that turns it into a
column is fit on the training split only (Notebook 08/09).

In [18]:
unknown_counts = {
    col: int((df[col] == "unknown").sum())
    for col in categorical_cols if "unknown" in df[col].unique()
}
print("'unknown' counts kept as-is (not imputed):")
for col, cnt in unknown_counts.items():
    print(f"  {col:12s}: {cnt:6,} ({cnt / len(df):.1%})")

log_clean("keep_unknown_as_category",
           f"Left 'unknown' as an explicit level in {list(unknown_counts)}, "
           "per Notebook 02 Section 6.2 (default's association was significant).")

'unknown' counts kept as-is (not imputed):
  job         :    325 (0.8%)
  marital     :     79 (0.2%)
  education   :  1,686 (4.3%)
  default     :  8,266 (21.0%)
  housing     :    980 (2.5%)
  loan        :    980 (2.5%)
[CLEAN] keep_unknown_as_category: Left 'unknown' as an explicit level in ['job', 'marital', 'education', 'default', 'housing', 'loan'], per Notebook 02 Section 6.2 (default's association was significant).


### 2.5 Defensive Consistency Check

Notebook 01 already established there's no inconsistent spelling and no
out-of-range numeric values. This is a fast, cheap re-check on the
post-cleaning frame not a re-audit since cleaning steps above touched
several columns and a silent bug here would be expensive to find later.

In [19]:
before_cols = df.shape[1]
df = engineer_features(df, leakage_cols=leakage_cols)

# The transformations themselves now live in src/features.py, shared with
# models/predict.py, so training and scoring can never silently drift apart.
# The log_* calls below just preserve the audit trail this notebook has
# always written to reports/clean_config.json.
log_clean("drop_raw_pdays",
           "Dropped raw pdays (sentinel-corrupted, low information density); replaced by "
           "contacted_before + pdays_known_days. Logic: src/features.py.")
log_fe("contacted_before", "binary",
       "previous>0, using the more reliable of the two prior-contact signals "
       "(100% agreement with poutcome; Notebook 01 v2 Section 7) instead of pdays.")
log_fe("pdays_known_days", "numeric (96.3% missing)",
       "pdays with the 999 sentinel set to NaN. Missingness is structural, not random -- "
       "it will be imputed on the training split only (Notebook 08/09), never here.")
for col in ["month_sin", "month_cos", "day_of_week_sin", "day_of_week_cos"]:
    log_fe(col, "numeric [-1, 1]",
           "Fixed sine/cosine transform of the calendar field -- deterministic, "
           "no fitting, captures cyclical adjacency a one-hot encoding would lose.")
log_clean("drop_raw_calendar_columns",
           "Dropped raw month, day_of_week after deriving their cyclical encodings "
           "(src/features.py).")
log_fe("education_ordinal", "numeric ordinal (with NaN for 'unknown')",
       "Fixed domain-knowledge order applied to education; 'unknown' rows are NaN here -- "
       "imputed on the training split only, downstream.")
log_clean("drop_raw_education",
           "Dropped raw education after deriving education_ordinal (src/features.py).")
log_fe("age_group", "categorical (nominal)",
       "Fixed banking-segment age buckets, offered as a candidate feature -- "
       "kept/dropped based on incremental value in Notebook 09, not asserted here.")

print(f"Columns after engineer_features(): {before_cols} -> {df.shape[1]}")
df[["contacted_before", "pdays_known_days", "month_sin", "month_cos",
    "day_of_week_sin", "day_of_week_cos", "education_ordinal", "age_group"]].head()


[CLEAN] drop_raw_pdays: Dropped raw pdays (sentinel-corrupted, low information density); replaced by contacted_before + pdays_known_days. Logic: src/features.py.
[FEATURE] contacted_before (binary): previous>0, using the more reliable of the two prior-contact signals (100% agreement with poutcome; Notebook 01 v2 Section 7) instead of pdays.
[FEATURE] pdays_known_days (numeric (96.3% missing)): pdays with the 999 sentinel set to NaN. Missingness is structural, not random -- it will be imputed on the training split only (Notebook 08/09), never here.
[FEATURE] month_sin (numeric [-1, 1]): Fixed sine/cosine transform of the calendar field -- deterministic, no fitting, captures cyclical adjacency a one-hot encoding would lose.
[FEATURE] month_cos (numeric [-1, 1]): Fixed sine/cosine transform of the calendar field -- deterministic, no fitting, captures cyclical adjacency a one-hot encoding would lose.
[FEATURE] day_of_week_sin (numeric [-1, 1]): Fixed sine/cosine transform of the calendar f

,contacted_before,pdays_known_days,month_sin,month_cos,day_of_week_sin,day_of_week_cos,education_ordinal,age_group
0,0,NaN,0.5,-0.866025,0.0,1.0,1.0,50-59
1,0,NaN,0.5,-0.866025,0.0,1.0,4.0,50-59
2,0,NaN,0.5,-0.866025,0.0,1.0,4.0,30-39
3,0,NaN,0.5,-0.866025,0.0,1.0,2.0,40-49
4,0,NaN,0.5,-0.866025,0.0,1.0,4.0,50-59


In [20]:
assert df[target_col].isin(["yes", "no"]).all(), "Unexpected target values."
assert df["age"].between(0, 120).all(), "Out-of-range age after cleaning."
assert (df["contacted_before"].isin([0, 1])).all(), "contacted_before not binary."
assert df.shape[0] == df.drop_duplicates().shape[0], "Duplicates remain after dedup step."
for col in categorical_cols:
    if col in df.columns:
        assert df[col].str.strip().eq(df[col]).all(), f"Untrimmed whitespace in {col}."
print("All defensive checks passed.")

All defensive checks passed.


## 3. Feature Engineering

Calendar cyclical encoding (3.1), education ordinal encoding (3.2), and
age buckets (3.3) are now implemented in `src/features.py` as part of
the shared `engineer_features()` function called above (Section 2.3),
rather than as separate cells here.

Why: this exact same function is imported and called by
`models/predict.py` when scoring new customers. Keeping one copy of
the logic means training and production scoring are guaranteed to
apply identical transformations -- previously these lived only in
this notebook, so `predict.py` had to assume its input was already
transformed correctly, which is exactly the kind of drift that caused
an earlier bug in this project.

See `src/features.py` for the exact logic and rationale docstrings for
each transformation:
- **3.1 Calendar cyclical encoding** -- `month`/`day_of_week` -> `month_sin/cos`,
  `day_of_week_sin/cos` (fixed sine/cosine formula, safe to compute before the split)
- **3.2 `education` ordinal encoding** -- domain-knowledge integer scale,
  `"unknown"` -> `NaN` (imputed downstream, training split only)
- **3.3 `age_group` buckets** -- fixed banking-segment age bins, a candidate
  feature for Notebook 09 to keep or drop


### 3.4 Macro-Economic Columns Deferred, Not Reduced Here

Notebook 02 (Section 7.2, 7.3) found `emp.var.rate`, `euribor3m`,
`nr.employed`, and `cons.price.idx` highly collinear (VIF up to 64) and
flagged a composite (PCA) as a candidate. **PCA is not fit in this
notebook** a PCA's components are learned from the data's covariance
structure, and fitting it on the full dataset before the split would leak
test-set structure into the transformation every row (including test rows)
receives. All four raw columns are kept as-is here; the PCA (or a simpler
drop-two-of-four rule) is fit on the training split only, in
`09_feature_selection`.

In [21]:
macro_cols = profile["macro_economic_cols"]
print(f"Macro-economic columns kept as raw, untouched: {macro_cols}")
print("Dimensionality reduction for this block is deferred to 09_feature_selection "
      "(fit on the training split only).")

Macro-economic columns kept as raw, untouched: ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Dimensionality reduction for this block is deferred to 09_feature_selection (fit on the training split only).


### 3.5 Leakage Sanity Check on New Features

A quick check that none of the newly engineered columns accidentally
became a near-perfect proxy for the target the way `duration` was cheap
insurance before this dataframe becomes the basis for every notebook after
it.

In [22]:
y_binary = (df[target_col] == "yes").astype(int)
new_numeric_cols = ["contacted_before", "pdays_known_days", "month_sin", "month_cos",
                     "day_of_week_sin", "day_of_week_cos", "education_ordinal"]

leak_check = pd.DataFrame({
    "Feature": new_numeric_cols,
    "Correlation_with_target": [df[c].corr(y_binary) for c in new_numeric_cols],
})
leak_check["Abs_Correlation"] = leak_check["Correlation_with_target"].abs()
leak_check = leak_check.sort_values("Abs_Correlation", ascending=False).reset_index(drop=True)
print(leak_check)

max_corr = leak_check["Abs_Correlation"].max()
assert max_corr < 0.40, (
    f"New feature correlation ({max_corr:.2f}) is in duration's leakage range (0.40+) -- investigate."
)
print(f"\nHighest new-feature correlation is {max_corr:.2f} -- well below duration's 0.41. No new leakage found.")

             Feature  Correlation_with_target  Abs_Correlation
0   contacted_before                 0.193174         0.193174
1          month_cos                 0.111238         0.111238
2  education_ordinal                 0.054988         0.054988
3   pdays_known_days                -0.035944         0.035944
4          month_sin                -0.027899         0.027899
5    day_of_week_cos                -0.022170         0.022170
6    day_of_week_sin                 0.005856         0.005856

Highest new-feature correlation is 0.19 -- well below duration's 0.41. No new leakage found.


## 4. Final Schema & Column Roles for the Downstream Pipeline

Every column from here is labelled by *how* it should be handled in the
`ColumnTransformer` built in Notebook 08/09 — this replaces re-deriving
column roles by hand in every later notebook.

In [23]:
column_roles = {
    "target": [target_col],
    "numeric_passthrough": ["age", "campaign", "previous", "emp.var.rate",
                             "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed",
                             "month_sin", "month_cos", "day_of_week_sin", "day_of_week_cos"],
    "numeric_needs_imputation": ["pdays_known_days", "education_ordinal"],
    "binary_passthrough": ["contacted_before"],
    "nominal_needs_encoding": ["job", "marital", "default", "housing", "loan",
                                "contact", "poutcome", "age_group"],
    "already_engineered_ordinal": ["education_ordinal"],
    "dropped_from_raw": leakage_cols + ["pdays", "month", "day_of_week", "education"],
}

for role, cols in column_roles.items():
    present = [c for c in cols if c in df.columns or role == "dropped_from_raw"]
    print(f"{role:28s}: {present}")

all_kept_cols = set(df.columns) - {target_col}
accounted_for = set(column_roles["numeric_passthrough"] + column_roles["numeric_needs_imputation"]
                     + column_roles["binary_passthrough"] + column_roles["nominal_needs_encoding"])
unaccounted = all_kept_cols - accounted_for
assert not unaccounted, f"Columns present in df but not classified: {unaccounted}"
print("\nEvery surviving column is accounted for in column_roles.")

target                      : ['y']
numeric_passthrough         : ['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos']
numeric_needs_imputation    : ['pdays_known_days', 'education_ordinal']
binary_passthrough          : ['contacted_before']
nominal_needs_encoding      : ['job', 'marital', 'default', 'housing', 'loan', 'contact', 'poutcome', 'age_group']
already_engineered_ordinal  : ['education_ordinal']
dropped_from_raw            : ['duration', 'pdays', 'month', 'day_of_week', 'education']

Every surviving column is accounted for in column_roles.


### 4.1 Before / After Summary

In [24]:
summary = pd.DataFrame({
    "Stage": ["Raw (Notebook 01)", "After cleaning + feature engineering"],
    "Rows": [profile["n_rows"], df.shape[0]],
    "Columns": [profile["n_columns"], df.shape[1]],
})
summary

,Stage,Rows,Columns
0,Raw (Notebook 01),41188,21
1,After cleaning + feature engineering,39404,24


## 5. Save Cleaned Dataset & Config

The cleaned-and-engineered dataframe is saved once here; every notebook
from `08_train_validation_test_split` onward loads this file rather than
re-running cleaning logic. `clean_config.json` records every decision made
above with its rationale, so a reviewer (or a future notebook) doesn't
have to re-read this whole notebook to know what happened to a column.

In [25]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
Path("../reports").mkdir(parents=True, exist_ok=True)

output_path = Path("../data/processed/bank_marketing_clean.csv")
df.to_csv(output_path, index=False)
output_hash = hashlib.sha256(output_path.read_bytes()).hexdigest()

clean_config = {
    "source_notebook": "03_data_cleaning_feature_engineering.ipynb",
    "input_sha256": profile["source_sha256"],
    "output_file": str(output_path),
    "output_sha256": output_hash,
    "n_rows": df.shape[0],
    "n_columns": df.shape[1],
    "target_col": target_col,
    "target_positive_label": "yes",
    "cleaning_log": cleaning_log,
    "feature_engineering_log": fe_log,
    "column_roles_for_pipeline": column_roles,
    "deferred_to_train_only_pipeline": [
        "One-hot encoding vocabulary for nominal_needs_encoding columns",
        "Imputation values for pdays_known_days and education_ordinal "
        "(numeric_needs_imputation)",
        "Any scaling (StandardScaler/MinMaxScaler) if the chosen model needs it",
        "PCA / dimensionality reduction on the macro-economic column block",
    ],
}

with open("../reports/clean_config.json", "w") as f:
    json.dump(clean_config, f, indent=2, default=str)

print(f"Saved -> {output_path}  ({df.shape[0]:,} rows x {df.shape[1]} columns)")
print(f"Saved -> ../reports/clean_config.json")
print(f"Output file SHA-256: {output_hash}")

Saved -> ..\data\processed\bank_marketing_clean.csv  (39,404 rows x 24 columns)
Saved -> ../reports/clean_config.json
Output file SHA-256: 9b069b524d740c3e0dc0c157e0a3d948f4977db6983628c22e49a41a5f7615de


## Conclusion

Cleaning removed the leakage column, resolved the duplicate-rows question
left open since Notebook 01 using the target-rate evidence gathered there,
and replaced the sentinel-corrupted `pdays` with a reliable
`contacted_before` flag plus a sparse supplementary day-count. Feature
engineering added cyclical calendar encodings and an ordinal education
scale, both fixed-formula transforms safe to apply before any split, and
explicitly deferred every data-fitted transformation (imputation, one-hot
vocabularies, scaling, PCA) to a pipeline that will be fit on the training
split only.

**Next notebook: `08_train_validation_test_split`** given Notebook 02
(Section 8.1) found the rows are chronologically ordered and showed the
target rate shifting with the macro-economic cycle, the split strategy
decided there needs to be time-aware, not a random shuffle; that notebook
also builds the `ColumnTransformer` pipeline this notebook set up column
roles for, fitting every data-dependent step on the training rows only.